## Framework Overview  

This framework is designed to streamline the **data preprocessing pipeline** by implementing multiple algorithms for:  

- **Data Cleansing**  
- **Outlier Detection & Handling**  
- **Feature Selection**  
- **Model Selection**  

At each step, we evaluate the effectiveness of different algorithms using a **shallow Decision Tree model** and  a **KNN Model**. This allows us to determine the best preprocessing strategy based on performance.  


## Importing Packages

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split #Weak Model Test
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score #Weak Model Test
from sklearn.tree import DecisionTreeClassifier #Weak Model Test
from sklearn.neighbors import KNeighborsClassifier #Weak Model Test
from sklearn.impute import SimpleImputer, KNNImputer #Handle Missing
from sklearn.linear_model import LinearRegression #Handle Missing
from sklearn.experimental import enable_iterative_imputer #Handle Missing(Imported Because of API Change)
from sklearn.impute import IterativeImputer #Handle Missing
from sklearn.neighbors import LocalOutlierFactor #Tame Outlier
from sklearn.ensemble import IsolationForest #Tame Outlier
from scipy.stats import zscore #Tame Outlier
from sklearn.preprocessing import RobustScaler,MinMaxScaler,StandardScaler # Nomalization
from sklearn.feature_selection import SelectKBest,f_classif,SelectFromModel # Feature Selection
from sklearn.svm import LinearSVC # Feature Selection
from sklearn.ensemble import ExtraTreesClassifier # Feature Selection

## Handle Date Columns

In [ ]:
def identify_date_columns(data):
    """ Detect date columns."""
    date_columns=[]
    date_pattern = r'(\d{1,2}[-/]\d{1,2}[-/]\d{2,4}|\d{4}[-/]\d{1,2}[-/]\d{1,2}|\d{1,2}[-/]\d{1,2}[-/]\d{2,4})'

    for col in data.columns:
        if data[col].dtype.name == 'object':
            if all(data[col].str.contains(date_pattern, regex=True, na=True)):
                date_columns.append(col)
    return date_columns

def process_date_columns(data, reference_column=None):
    """ Processes the date columns and extracts useful time-based features."""

    date_columns=identify_date_columns(data)

    for col in date_columns:
        data[col] = pd.to_datetime(data[col], errors='coerce')

    for col in date_columns:
        data[f'{col}_year'] = data[col].dt.year.astype(float)
        data[f'{col}_month'] = data[col].dt.month.astype(float)
        data[f'{col}_day'] = data[col].dt.day.astype(float)
        data[f'{col}_day_of_week'] = data[col].dt.dayofweek.astype(float)
        data[f'{col}_hour'] = data[col].dt.hour.astype(float)
        data[f'{col}_day_of_year'] = data[col].dt.dayofyear.astype(float)
        data[f'{col}_is_weekend'] = (data[col].dt.weekday >= 5).astype(float)


    if reference_column:
        for col in date_columns:
            data[f'{col}_time_diff'] = (data[reference_column] - data[col]).dt.total_seconds() / (60 * 60 * 24)

    return data.drop(columns=date_columns)

## Convert into Numeric Values

In [ ]:
def ConvertToNumeric(data):
    data=process_date_columns(data)
    cols = data.columns
    num_cols = data._get_numeric_data().columns
    categorical_columns = list(set(cols) - set(num_cols))

    for column in categorical_columns:
        categories = list(data[column].dropna().astype(str).unique())
        data[column] = data[column].map(lambda x: categories.index(str(x)) if str(x) in categories else x)

    return data

## Loading Data

In [ ]:
def load_data(path, y_column):
    if("csv" in path):
        data=pd.read_csv(path)
    elif("xlsx" in path):
        data=pd.read_excel(path)
    else:
        print("Format not Supported")
        return None
    data=ConvertToNumeric(data)
    Y=data[y_column]
    X=data.drop(columns=y_column)
    return X,Y

## Implementing Test Models

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

def train_and_evaluate(X, y, expand=False):
    """Trains multiple weak models (3x KNN and 3x Decision Tree) and evaluates them."""

    if len(y) <= 5 or len(X) <= 5:
        return 0

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    models = [
        KNeighborsClassifier(n_neighbors=3),
        KNeighborsClassifier(n_neighbors=5),
        KNeighborsClassifier(n_neighbors=7),

        DecisionTreeClassifier(max_depth=3, random_state=42),
        DecisionTreeClassifier(max_depth=5, random_state=42),
        DecisionTreeClassifier(max_depth=None, min_samples_split=10, random_state=42),

    ]

    all_scores = []

    for i, model in enumerate(models):
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average='weighted', zero_division=1)
        rec = recall_score(y_test, y_pred, average='weighted', zero_division=1)
        f1 = f1_score(y_test, y_pred, average='weighted')

        all_scores.extend([acc, prec, rec, f1])

        if expand:
            print(f"\nModel {i+1}: {model.__class__.__name__}")
            print(f"  Accuracy: {acc:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}, F1-score: {f1:.4f}")

    weighted_score = sum(all_scores) / len(all_scores)

    return weighted_score


# Handling Missing Data

These functions help clean our dataset by handling missing values efficiently and selecting the best imputation method.

## Available Algorithms  
- **Dropping Methods:**  
  - Drop columns with null values exceeding a threshold  
  - Drop rows with missing values  

- **Statistical Imputation:**  
  - Impute with **mean** or **median**  
  - Impute with **class-specific mean** or **median**  

- **Forward & Backward Filling:**  
  - **Forward fill (ffill)**  
  - **Backward fill (bfill)**  
  - **Interpolate** missing values  

- **Model-Based Imputation:**  
  - **Model Imputation** (predict missing values using a simple model)  
  - **Iterative Model Imputation** (refines predictions iteratively)  
  - **KNN Imputation** (fills missing values based on k-nearest neighbors)  


In [ ]:
def drop_rows(X, Y):
    """Drops rows with any missing values."""

    mask=X.notna().all(axis=1)
    return X[mask], Y[mask]

def impute_mean(X, Y):
    """Fills missing values with column mean while leaving non-missing values intact."""

    imputer = SimpleImputer(strategy="mean")
    X_imputed = imputer.fit_transform(X)
    X_imputed_df = pd.DataFrame(X_imputed, columns=X.columns)

    X_filled = X.copy()
    for col in X.columns:
        X_filled[col] = np.where(X[col].isna(), X_imputed_df[col], X_filled[col])

    return X_filled,Y

def impute_median(X, Y):
    """Fills missing values with column median while leaving non-missing values intact."""

    imputer = SimpleImputer(strategy="median")
    X_imputed = imputer.fit_transform(X)
    X_imputed_df = pd.DataFrame(X_imputed, columns=X.columns)

    X_filled = X.copy()
    for col in X.columns:
        X_filled[col] = np.where(X[col].isna(), X_imputed_df[col], X_filled[col])

    return X_filled,Y

def impute_class_mean(X, Y):
    """Fills missing values with the mean of each class in Y, ensuring no NaNs remain."""

    X_copy = X.copy()
    for col in X.columns:
        grouped_means = X.groupby(Y)[col].transform(lambda x: x.mean())
        X_copy[col] = X[col].fillna(grouped_means)  # Fill NaNs with class mean

        if X_copy[col].isna().sum() > 0:
            X_copy[col].fillna(X[col].mean(), inplace=True)

    return X_copy, Y

def impute_class_median(X, Y):
    """Fills missing values with the median of each class in Y."""

    X_copy = X.copy()
    for col in X.columns:
        X_copy[col] = X.groupby(Y)[col].transform(lambda x: x.fillna(x.median()))

        if X_copy[col].isna().sum() > 0:
            X_copy[col].fillna(X[col].mean(), inplace=True)
    return X_copy,Y

def ffill(X, Y):
    """Fills missing values with the previous row (forward fill) and ensures no NaNs remain."""

    X_filled = X.ffill()
    X_filled = X_filled.bfill()
    return X_filled, Y

def bfill(X, Y):
    """Fills missing values with the next row (backward fill)."""

    X_filled = X.bfill()
    X_filled = X_filled.ffill()
    return X_filled,Y

def interpolate(X, Y):
    """Interpolates missing values linearly and ensures no NaNs remain."""

    X_filled = X.interpolate(method="linear", limit_direction="both")
    return X_filled, Y



def Model_imputation(X, Y):
    """Uses a simple regression model to impute missing values."""

    X_copy = X.copy()

    for col in X.columns:
        missing_mask = X_copy[col].isna()
        if missing_mask.sum() > 0:
            known_data = X_copy.dropna()
            known_X = known_data.drop(columns=[col])
            known_y = known_data[col]
            missing_X = X_copy.loc[missing_mask].drop(columns=[col])

            if missing_X.isna().sum().sum() > 0:
                missing_X = missing_X.fillna(known_X.mean())

            if len(known_X) > 0 and len(missing_X) > 0:
                model = LinearRegression()
                model.fit(known_X, known_y)
                X_copy.loc[missing_mask, col] = model.predict(missing_X)

    return X_copy, Y

def Iterative_model_Imputation(X, Y):
    """Uses iterative imputation (sklearn's IterativeImputer) to fill only null values."""

    imputer = IterativeImputer()
    X_imputed = imputer.fit_transform(X)
    X_imputed_df = pd.DataFrame(X_imputed, columns=X.columns)

    X_filled = X.copy()

    for col in X.columns:
        X_filled[col] = np.where(X[col].isna(), X_imputed_df[col], X_filled[col])

    return X_filled,Y

def KNN_Imputation(X, Y, n_neighbors=5):
    """Uses KNN to fill missing values while leaving non-missing values intact."""

    imputer = KNNImputer(n_neighbors=n_neighbors)
    X_imputed = imputer.fit_transform(X)
    X_imputed_df = pd.DataFrame(X_imputed, columns=X.columns)

    X_filled = X.copy()

    for col in X.columns:
        X_filled[col] = np.where(X[col].isna(), X_imputed_df[col], X_filled[col])

    return X_filled,Y

algorithm_functions_clean_data = {
    "drop_rows": drop_rows,
    "impute_mean": impute_mean,
    "impute_median": impute_median,
    "impute_class_mean": impute_class_mean,
    "impute_class_median": impute_class_median,
    "ffill": ffill,
    "bfill": bfill,
    "interpolate": interpolate,
    "Model_imputation": Model_imputation,
    "Iterative_model_Imputation": Iterative_model_Imputation,
    "KNN_Imputation": KNN_Imputation,
}


In [ ]:
def handling_missing_data(X, Y):
    """Applies different missing data handling algorithms, selects the best one, and returns the transformed dataset."""
    if(X.isna().sum().sum()==0):
        return X,Y,"No Null Data",100,{"No Null Data":100}
    Y = Y.dropna(inplace=False)
    X = X.loc[Y.index]
    x_copy = X.copy()
    y_copy = Y.copy()
    acc_holder = {}

    best_algo = None
    best_accuracy = -1

    for name, func in algorithm_functions_clean_data.items():
        X_transformed,Y_transformed = func(x_copy.copy(), y_copy.copy())
        print(name)
        print()
        accuracy = train_and_evaluate(X_transformed, Y_transformed,expand=True)
        acc_holder[name] = accuracy

        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_algo = name
    if best_algo:
        x_copy,y_copy = algorithm_functions_clean_data[best_algo](x_copy.copy(), y_copy.copy())

    return x_copy, y_copy, best_algo, best_accuracy,acc_holder

# Taming Outliers

These functions help clean our dataset from outliers efficiently and selecting the best outlier detection algorithm.

## Available Algorithms  
- **IQR Method(1.5):**  
  - Take 1.5 times the IQR and then subtract this value from Q1 and add this value to Q3


- **LOF:**  
  - For any data object **q**, the **LOF score** is computed as the ratio of the **average local density** of its **k-nearest neighbors** to its **own local density** **[25]**.  

  $$
  LOF(q) = \frac{\sum_{x \in N_k(q)} lrd(x)}{|N_k(q)| \times lrd(q)}
  $$

  where the **local reachability density (lrd)** of **q** is given by:  

  $$
  lrd(q) = \frac{|N_k(q)|}{\sum_{x \in N_k(q)} \max(\text{dist}_k(x, D), \text{dist}(q, x))}
  $$

- **SP:**  
   - Employ a **scoring measure** based on the nearest neighbor (**k = 1**) within random sub-samples (**S ⊂ D**).  

    $$  S_p(q) = \min_{{x \in S}} \text{dist}(q, x) $$

    where **dist(q, x)** represents the distance between **q** and **x**.

- **iForest:**  
  - A **random split** is performed on a randomly selected feature.  
  - The partitioning continues until either:  
    - Each node contains only **one data object**, or  
    - The tree reaches its **height limit**.

  $$
  iForest(q) = \frac{1}{t} \sum_{i=1}^{t} l_i(q)
  $$


- **iNNe:**  
  - This method builds **hyperspheres** using all dimensions of the dataset. The **isolation score** of a data object **q** is defined as:  

  $$
  I(q) =
  \begin{cases}
  \tau (\eta_{cnn}(q)), & \text{if } q \in \bigcup_{c \in S} B(c) \\  
  1 - \tau (cnn(q)), & \text{otherwise}  
  \end{cases}
  $$


In [ ]:
def IQR(X, Y):
    """Adjust outliers based on IQR min-max whiskers."""
    Q1 = X.quantile(0.25)
    Q3 = X.quantile(0.75)
    IQR = Q3 - Q1
    min_whisker = Q1 - 1.5 * IQR
    max_whisker = Q3 + 1.5 * IQR
    X_adjusted = X.clip(lower=min_whisker, upper=max_whisker, axis=1)
    return X_adjusted, Y

def LOF(X, Y):
    """Adjust only the detected outliers based on LOF anomaly score."""
    clf = LocalOutlierFactor(n_neighbors=20, contamination=0.1)
    X_scores = clf.fit_predict(X)
    adjustment_factor = np.abs(clf.negative_outlier_factor_) / np.max(np.abs(clf.negative_outlier_factor_))

    X_adjusted = X.copy()
    outlier_mask = X_scores == -1
    X_adjusted[outlier_mask] = X[outlier_mask] * (1 - adjustment_factor[outlier_mask, np.newaxis])
    return X_adjusted, Y

def IsolationForestOutlier(X, Y):
    """Only adjust outliers based on Isolation Forest isolation score."""
    iso = IsolationForest(n_estimators=100, contamination=0.1, random_state=42)
    preds = iso.fit_predict(X)  # -1 for outliers, 1 for inliers
    scores = iso.decision_function(X)

    adjustment_factor = (scores - scores.min()) / (scores.max() - scores.min())
    X_adjusted = X.copy()

    for i, pred in enumerate(preds):
        if pred == -1:
            X_adjusted.iloc[i] = X.iloc[i] * (1 - adjustment_factor[i])

    return X_adjusted, Y



def SP(X, Y):
    """Adjust only outliers beyond 3 std dev using Standardization Projection."""
    X_adjusted = X.copy()
    Z_scores = np.abs(zscore(X, nan_policy='omit'))
    Z_scores = np.nan_to_num(Z_scores, nan=0)

    for col in X.columns:
        col_idx = X.columns.get_loc(col)
        for i in range(len(X)):
            if Z_scores[i, col_idx] > 3:
                factor = 3 / Z_scores[i, col_idx]
                X_adjusted.iloc[i, col_idx] = X.iloc[i, col_idx] * factor

    return X_adjusted, Y


def IsolationNNe(X, Y):
    """Only adjust outliers based on Isolation Forest distance score."""
    iso = IsolationForest(contamination=0.05, n_estimators=200, random_state=42)
    preds = iso.fit_predict(X)
    dist = iso.decision_function(X)
    adjustment_factor = np.abs(dist) / np.max(np.abs(dist))

    X_adjusted = X.copy()
    for i, pred in enumerate(preds):
        if pred == -1:
            X_adjusted.iloc[i] = X.iloc[i] * (1 - adjustment_factor[i])

    return X_adjusted, Y


algorithm_functions_tame_outlier = {
    "IQR": IQR,
    "LOF": LOF,
    "iForest": IsolationForestOutlier,
    "SP": SP,
    "iNNe": IsolationNNe,
}


In [ ]:
def taming_outliers(X,Y):
    acc_holder = {}
    x_copy = X.copy()
    y_copy = Y.copy()
    best_algo = None
    best_accuracy = 0


    for name, func in algorithm_functions_tame_outlier.items():
        X_Adjusted,Y_Adjusted = func(x_copy.copy(), y_copy.copy())
        print(name)
        print()
        accuracy = train_and_evaluate(X_Adjusted, Y_Adjusted,expand=True)
        acc_holder[name] = accuracy


        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_algo = name
    if best_algo:
        x_copy,y_copy = algorithm_functions_tame_outlier[best_algo](x_copy.copy(), y_copy.copy())

    return x_copy, y_copy, best_algo, best_accuracy,acc_holder

# Normalization Algorithms

These functions help standardize or normalize datasets to improve model performance and feature scaling.

## Available Algorithms

### 1. MinMaxScalarNorm
- **Description:**
  - Scales features to a fixed range [0, 1] using minimum and maximum values
  - Formula: 
    $$
    X_norm = (X - X_min) / (X_max - X_min)
    $$


### 2. RobustScalarNorm
- **Description:**
  - Scales features using median and interquartile range (IQR) to handle outliers
  - Formula:
    $$
    X_robust = (X - Median(X)) / IQR(X)
    $$


### 3. ZScoreNormalizationNorm
- **Description:
  - Standardizes features to have zero mean and unit variance
  - Formula:
    $$
    X_std = (X - μ) / σ
    $$


In [ ]:
def MinMaxScalarNorm(X,Y):
  scaler = MinMaxScaler()
  scaler.fit(X)
  return scaler.transform(X)

def RobustScalarNorm(X,Y):
  transformer=RobustScaler(with_centering=False,copy=False,unit_variance=True,with_scaling=False).fit(X)
  return transformer.transform(X)

def ZScoreNormalizationNorm(X,Y):
    scaler = StandardScaler()
    return scaler.fit_transform(X)

algorithm_functions_normalization = {
    "MinMaxScalarNorm": MinMaxScalarNorm,
    "RobustScalarNorm": RobustScalarNorm,
    "ZScoreNormalizationNorm": ZScoreNormalizationNorm,
}

In [ ]:
def normalization(X,Y):
    acc_holder = {}
    x_copy = X.copy()
    y_copy = Y.copy()
    best_algo = None
    best_accuracy = 0


    for name, func in algorithm_functions_normalization.items():
        X_Adjusted,Y_Adjusted = func(x_copy.copy(), y_copy.copy())
        print(name)
        print()
        accuracy = train_and_evaluate(X_Adjusted, Y_Adjusted,expand=True)
        acc_holder[name] = accuracy


        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_algo = name
    if best_algo:
        x_copy,y_copy = algorithm_functions_normalization[best_algo](x_copy.copy(), y_copy.copy())

    return x_copy, y_copy, best_algo, best_accuracy,acc_holder

# Feature Selection Algorithms

These functions help select the most relevant features from a dataset to improve model performance and reduce dimensionality.

## Available Algorithms

### 1. $ SelectK $
- **Description:**
  - Selects the top K features based on statistical tests (chi2, f_classif, etc.)
  - Uses univariate statistical tests to score each feature


- **Parameters:**
  - k: Number of top features to select
  - score_func: Scoring function (default: f_classif)

### 2. $ L_1 Based $ 
- **Description:**
  - Selects features using L1-regularized linear models (Lasso)
  - Features with non-zero coefficients are selected


- **Parameters**:
  - estimator: L1-regularized model (e.g., LogisticRegression, Lasso)
  - threshold: Minimum coefficient value for selection

### 3. $ TreeBased $
- **Description:**
  - Selects features based on importance scores from tree-based models
  - Uses feature_importances_ attribute from tree models


- **Parameters**:
  - estimator: Tree-based model (e.g., RandomForest, XGBoost)
  - threshold: Minimum importance score for selection

In [ ]:
def SelectK(X, Y):
    selector = SelectKBest(f_classif, k="all")
    selector.fit(X.values, Y)
    selected_X = selector.transform(X.values)
    scores = selector.scores_
    return pd.DataFrame(selected_X, columns=X.columns[selector.get_support()]), Y, scores


def L1_Based(X, Y):
    lsvc = LinearSVC(C=0.01, penalty="l1", dual=False, max_iter=2000).fit(X.values, Y)
    model = SelectFromModel(lsvc, prefit=True)
    selected_X = model.transform(X.values)
    coefs = np.abs(lsvc.coef_).mean(axis=0)
    return pd.DataFrame(selected_X, columns=X.columns[model.get_support()]), Y, coefs


def TreeBased(X, Y):
    clf = ExtraTreesClassifier(n_estimators=50)
    clf.fit(X.values, Y)
    model = SelectFromModel(clf, prefit=True)
    selected_X = model.transform(X.values)
    importances = clf.feature_importances_
    return pd.DataFrame(selected_X, columns=X.columns[model.get_support()]), Y, importances


algorithm_functions_feature_selection = {
    "SelectK": SelectK,
    "L1_Based": L1_Based,
    "TreeBased": TreeBased,
}

In [ ]:
def remove_constant_features(X):
    """Removes features that have the same value in all samples."""
    return X.loc[:, X.nunique() > 1]

def feature_selection(X,Y):
    acc_holder = {}
    x_copy = X.copy()
    y_copy = Y.copy()
    best_algo = None
    best_accuracy = 0
    final_scores=np.zeros(len(remove_constant_features(X).columns))
    for name, func in algorithm_functions_feature_selection.items():
        X_Adjusted,Y_Adjusted,Scores = func(remove_constant_features(x_copy.copy()), y_copy.copy())
        print(name)
        print()

        final_scores+=((Scores - Scores.min()) / (Scores.max() - Scores.min()))

        accuracy = train_and_evaluate(X_Adjusted, Y_Adjusted,expand=True)
        acc_holder[name] = accuracy

        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_algo = name

    if best_algo:
        _,_,Scores = algorithm_functions_feature_selection[best_algo](remove_constant_features(x_copy.copy()), y_copy.copy())
        final_scores+=Scores/2

    percent = 0.5
    k = int(len(final_scores) * percent)
    top_column_indices = np.argsort(final_scores)[-k:][::-1]
    selected_columns = X.columns[top_column_indices]
    x_copy = x_copy[selected_columns]

    return x_copy, y_copy, best_algo, best_accuracy,acc_holder

In [ ]:
# X,Y=load_data(path='./drive/MyDrive/Shariati_final_results.xlsx',y_column='کووید 19-RT-PCR(COVID-19)')
# X,Y=load_data(path="data1_1.csv",y_column="smokingstutus")
X,Y=load_data(path="./drive/MyDrive/healthcare_dataset.csv",y_column="Test Results")

In [ ]:
x_copy_clean_data, y_copy_clean_data, best_algo_clean_data, best_accuracy_clean_data,acc_holder_clean_data=handling_missing_data(X,Y)

In [ ]:
x_copy_tame_outlier, y_copy_tame_outlier, best_algo_tame_outlier, best_accuracy_tame_outlier,acc_holder_tame_outlier=taming_outliers(x_copy_clean_data,y_copy_clean_data)

In [ ]:
x_copy_normalization, y_copy_normalization, best_algo_normalization, best_accuracy_normalization,acc_holder_normalization=normalization(x_copy_tame_outlier,y_copy_tame_outlier)

In [ ]:
x_copy_feature_selection, y_copy_feature_selection, best_algo_feature_selection, best_accuracy_feature_selection,acc_holder_feature_selection=feature_selection(x_copy_tame_outlier,y_copy_tame_outlier)